# Notebook 05b: Baseline Models with Pipeline

---

## Overview

This notebook demonstrates a **production-grade ML pipeline** for baseline disease classification.

**Differences from 05a (Naive Approach):**

| Aspect | 05a Naive | 05b Pipeline |
|--------|-----------|---------------|
| **Feature extraction** | Manual loops in notebook | sklearn transformers |
| **Composability** | Hard-coded functions | Configurable FeatureUnion |
| **Reproducibility** | Notebook-specific | Config-driven, portable |
| **Persistence** | Pickle whole pipeline | joblib + JSON (best practice) |
| **Parallelism** | Sequential | FeatureUnion(n_jobs=-1) |
| **Testing** | None | Unit testable components |
| **Hyperparameter tuning** | Manual edits | GridSearchCV-ready |

**Key Benefits:**
- ✅ **Reproducible**: Config-driven, deterministic
- ✅ **Portable**: Works outside notebooks (CLI, API)
- ✅ **Scalable**: FeatureUnion parallelizes feature extraction
- ✅ **Maintainable**: Modular, testable components
- ✅ **Best practices**: No pickle, version tracking, manifest

---

## 1. Setup

In [ ]:
# Standard library
import json
import sys
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Our ML pipeline package
sys.path.insert(0, str(Path.cwd().parent))
from src.ml import build_pipeline, load_config, save_pipeline, load_pipeline, get_manifest

print("✓ Imports successful")

In [ ]:
# Define paths
current_path = Path.cwd()

if current_path.name == 'jupyter_notebooks':
    PROJECT_ROOT = current_path.parent
elif (current_path / 'setup.py').exists() or (current_path / 'README.md').exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models' / 'saved_models'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

print(f"Project root: {PROJECT_ROOT}")
print(f"Config directory: {CONFIGS_DIR}")

## 2. Load Configuration

In [ ]:
# Load experiment config
config_path = CONFIGS_DIR / 'baseline_experiment.yaml'
config = load_config(config_path)

print("✓ Configuration loaded")
print(f"\nExperiment: {config['experiment']['name']} {config['experiment']['version']}")
print(f"Description: {config['experiment']['description']}")
print(f"\nKey parameters:")
print(f"  Image size: {config['image']['img_size']}")
print(f"  HOG orientations: {config['hog']['orientations']}")
print(f"  GLCM levels: {config['glcm']['levels']}")
print(f"  XGBoost estimators: {config['xgb']['n_estimators']}")
print(f"  Random state: {config['runtime']['random_state']}")
print(f"  Sampling enabled: {config['sampling']['enabled']}")

## 3. Load Data

In [ ]:
# Load split files from Notebook 03
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

print(f"✓ Loaded splits:")
print(f"  Train: {len(train_df):,} images")
print(f"  Val:   {len(val_df):,} images")
print(f"  Test:  {len(test_df):,} images")

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']
print(f"\n✓ Disease classes: {len(disease_classes)}")

In [ ]:
# Optionally sample for faster experimentation
if config['sampling']['enabled']:
    train_size = config['sampling']['train_size']
    val_size = config['sampling']['val_size']
    test_size = config['sampling']['test_size']
    random_state = config['runtime']['random_state']
    
    train_df = train_df.sample(n=min(train_size, len(train_df)), random_state=random_state)
    val_df = val_df.sample(n=min(val_size, len(val_df)), random_state=random_state)
    test_df = test_df.sample(n=min(test_size, len(test_df)), random_state=random_state)
    
    print(f"⚠️ Using sample mode for faster experimentation:")
    print(f"  Train: {len(train_df):,} images")
    print(f"  Val:   {len(val_df):,} images")
    print(f"  Test:  {len(test_df):,} images")
    print(f"\n  💡 Set sampling.enabled=false in config to use full dataset")
else:
    print("Using full dataset")

In [ ]:
# Prepare data for pipeline
# Pipeline expects: X = image paths (1D array), y = labels (2D array)

X_train = train_df['full_path'].values
y_train = train_df[disease_classes].values

X_val = val_df['full_path'].values
y_val = val_df[disease_classes].values

X_test = test_df['full_path'].values
y_test = test_df[disease_classes].values

print("✓ Data prepared for pipeline")
print(f"  X_train shape: {X_train.shape} (image paths)")
print(f"  y_train shape: {y_train.shape} ({len(disease_classes)} diseases)")

## 4. Build Pipeline

The pipeline factory creates:
```
Pipeline(
  prep: Pipeline(
    features: FeatureUnion[
      hog: HOGFeatures(...),
      glcm: GLCMTexture(...),
      stats: StatisticalPixels(...)
    ],
    scaler: StandardScaler()
  ),
  clf: MultiOutputClassifier(XGBClassifier(...))
)
```

In [ ]:
# Build pipeline from config
pipeline = build_pipeline(config)

print("✓ Pipeline built")
print(f"\nPipeline steps:")
for name, step in pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

print(f"\nPreprocessor steps:")
prep = pipeline.named_steps['prep']
for name, step in prep.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

print(f"\nFeature extractors:")
feature_union = prep.named_steps['features']
for name, extractor in feature_union.transformer_list:
    print(f"  {name}: {type(extractor).__name__}")

## 5. Train Pipeline

The pipeline handles everything:
1. Load images from paths
2. Extract HOG + GLCM + Stats features in parallel
3. Concatenate features
4. Scale features
5. Train 14 XGBoost binary classifiers

In [ ]:
print("="*60)
print("TRAINING PIPELINE")
print("="*60)
print(f"Training on {len(X_train):,} images...")
print(f"(This may take several minutes)")

# Train entire pipeline
pipeline.fit(X_train, y_train)

print("\n✓ Training complete!")

## 6. Evaluate Pipeline

In [ ]:
def evaluate_multi_label_model(pipeline, X, y, disease_names):
    """
    Evaluate multi-label classification performance.
    """
    # Get predictions
    y_pred = pipeline.predict(X)
    y_proba = pipeline.predict_proba(X)
    
    # For MultiOutputClassifier, predict_proba returns list of arrays
    # We need probabilities for class=1 for each output
    y_proba_positive = np.array([proba[:, 1] for proba in y_proba]).T
    
    results = {}
    
    for i, disease in enumerate(disease_names):
        y_true_disease = y[:, i]
        y_pred_disease = y_pred[:, i]
        y_proba_disease = y_proba_positive[:, i]
        
        # Calculate metrics
        accuracy = accuracy_score(y_true_disease, y_pred_disease)
        precision = precision_score(y_true_disease, y_pred_disease, zero_division=0)
        recall = recall_score(y_true_disease, y_pred_disease, zero_division=0)
        f1 = f1_score(y_true_disease, y_pred_disease, zero_division=0)
        
        # AUC-ROC (only if both classes present)
        if len(np.unique(y_true_disease)) > 1:
            auc = roc_auc_score(y_true_disease, y_proba_disease)
        else:
            auc = np.nan
        
        results[disease] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc
        }
    
    return results


print("✓ Evaluation function defined")

In [ ]:
print("="*60)
print("VALIDATION SET PERFORMANCE")
print("="*60)

val_results = evaluate_multi_label_model(pipeline, X_val, y_val, disease_classes)

print(f"\n{'Disease':<20} {'AUC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print("-"*60)

for disease in disease_classes:
    metrics = val_results[disease]
    print(f"{disease:<20} {metrics['auc']:>8.3f} {metrics['f1']:>8.3f} "
          f"{metrics['precision']:>10.3f} {metrics['recall']:>8.3f}")

# Calculate averages
avg_auc = np.mean([val_results[d]['auc'] for d in disease_classes if not np.isnan(val_results[d]['auc'])])
avg_f1 = np.mean([val_results[d]['f1'] for d in disease_classes])

print("-"*60)
print(f"{'AVERAGE':<20} {avg_auc:>8.3f} {avg_f1:>8.3f}")

print(f"\n✓ Validation complete")

## 7. Test Set Evaluation

In [ ]:
print("="*60)
print("TEST SET PERFORMANCE")
print("="*60)

test_results = evaluate_multi_label_model(pipeline, X_test, y_test, disease_classes)

print(f"\n{'Disease':<20} {'AUC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print("-"*60)

for disease in disease_classes:
    metrics = test_results[disease]
    print(f"{disease:<20} {metrics['auc']:>8.3f} {metrics['f1']:>8.3f} "
          f"{metrics['precision']:>10.3f} {metrics['recall']:>8.3f}")

# Calculate averages
test_avg_auc = np.mean([test_results[d]['auc'] for d in disease_classes if not np.isnan(test_results[d]['auc'])])
test_avg_f1 = np.mean([test_results[d]['f1'] for d in disease_classes])

print("-"*60)
print(f"{'AVERAGE':<20} {test_avg_auc:>8.3f} {test_avg_f1:>8.3f}")

print(f"\n✓ Test evaluation complete")

## 8. Save Pipeline (Best Practices)

Implements the spec:
- Preprocessor saved with `joblib`
- XGBoost models saved as JSON (portable)
- Manifest with metadata and versions

In [ ]:
# Prepare output directory
experiment_name = config['experiment']['name']
experiment_version = config['experiment']['version']
model_dir = MODELS_DIR / f"{experiment_name}_{experiment_version}"

# Prepare metrics for manifest
metrics = {
    'validation': {
        'avg_auc': float(avg_auc),
        'avg_f1': float(avg_f1),
        'per_disease': {k: {kk: float(vv) if not np.isnan(vv) else None 
                           for kk, vv in v.items()} 
                       for k, v in val_results.items()}
    },
    'test': {
        'avg_auc': float(test_avg_auc),
        'avg_f1': float(test_avg_f1),
        'per_disease': {k: {kk: float(vv) if not np.isnan(vv) else None 
                           for kk, vv in v.items()} 
                       for k, v in test_results.items()}
    }
}

# Save pipeline
save_pipeline(
    pipeline=pipeline,
    output_dir=model_dir,
    disease_classes=disease_classes,
    config=config,
    metrics=metrics
)

print(f"\n💾 Model saved to: {model_dir}")

## 9. Load and Verify Pipeline

In [ ]:
print("="*60)
print("TESTING SAVE/LOAD ROUND-TRIP")
print("="*60)

# Load pipeline from disk
loaded_pipeline = load_pipeline(model_dir)

# Get manifest
manifest = get_manifest(model_dir)
print(f"\nManifest info:")
print(f"  Model: {manifest['model_type']}")
print(f"  Base estimator: {manifest['base_estimator']}")
print(f"  Outputs: {manifest['n_outputs']}")
print(f"  XGBoost version: {manifest['versions']['xgboost']}")
print(f"  Validation AUC: {manifest['metrics']['validation']['avg_auc']:.3f}")
print(f"  Test AUC: {manifest['metrics']['test']['avg_auc']:.3f}")

In [ ]:
# Verify predictions are identical
print("\nVerifying loaded pipeline produces identical predictions...")

# Sample a few test images
sample_indices = np.random.choice(len(X_test), size=min(10, len(X_test)), replace=False)
X_sample = X_test[sample_indices]
y_sample = y_test[sample_indices]

# Original pipeline predictions
original_pred = pipeline.predict(X_sample)
original_proba = np.array([proba[:, 1] for proba in pipeline.predict_proba(X_sample)]).T

# Loaded pipeline predictions
loaded_pred = loaded_pipeline.predict(X_sample)
loaded_proba = np.array([proba[:, 1] for proba in loaded_pipeline.predict_proba(X_sample)]).T

# Check if identical
pred_match = np.array_equal(original_pred, loaded_pred)
proba_close = np.allclose(original_proba, loaded_proba, rtol=1e-5)

print(f"  Predictions match: {pred_match} ✓" if pred_match else f"  Predictions match: {pred_match} ✗")
print(f"  Probabilities close: {proba_close} ✓" if proba_close else f"  Probabilities close: {proba_close} ✗")

if pred_match and proba_close:
    print("\n✓ Save/load round-trip successful! Pipeline is deterministic.")
else:
    print("\n⚠️ Predictions differ after loading!")

## 10. Summary

### What We Built

A **production-grade ML pipeline** that:
1. ✅ Works directly on image paths (no manual feature extraction)
2. ✅ Uses sklearn-compatible transformers (testable, reusable)
3. ✅ Parallelizes feature extraction with FeatureUnion
4. ✅ Config-driven (reproducible, tunable)
5. ✅ Best-practice persistence (joblib + JSON, no pickle)
6. ✅ Portable (works outside notebooks)

### Performance Comparison

**05a Naive vs 05b Pipeline:**
- Both use identical feature extraction logic (HOG, GLCM, Stats)
- Both use XGBoost with same hyperparameters
- **Expected**: Similar performance metrics
- **Advantage of Pipeline**: Better engineering, not better ML

### Next Steps

This pipeline is ready for:
- 🔧 **Hyperparameter tuning**: GridSearchCV/RandomizedSearchCV
- 🧪 **Unit testing**: pytest for transformers and pipeline
- 🖥️ **CLI interface**: `python -m src.ml.cli train --config configs/...`
- 📊 **Experiment tracking**: MLflow, Weights & Biases integration
- 🚀 **Deployment**: API serving, batch inference

---

**Key Insight**: Good ML engineering isn't about better models - it's about **reproducible**, **maintainable**, **scalable** systems.

The naive notebook approach (05a) was great for learning. The pipeline approach (05b) is great for production.